# 01 - Project LiDAR on an RGB image

This notebook follows one KITTI frame from its raw Velodyne point cloud to image-space pixels. Its purpose is to verify the calibration and projection before building any sparse maps or training a model.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Launch Jupyter from the repository root or notebooks/.")
sys.path.insert(0, str(ROOT / "src"))

from rgb_lidar_fusion.calibration import parse_kitti_calibration_file
from rgb_lidar_fusion.project_lidar import load_velodyne_bin, project_lidar_to_image

## Select a frame

`DATA_ROOT` must contain the KITTI Object `image_2/`, `velodyne/`, and `calib/` directories. Projection is performed at the original image resolution so the calibration file can be used without rescaling.

In [ ]:
DATA_ROOT = ROOT / "data/kitti/training"
FRAME_INDEX = 0
MAX_DEPTH_M = 80.0
POINT_SIZE = 3.0

image_paths = sorted((DATA_ROOT / "image_2").glob("*.png"))
if not image_paths:
    raise FileNotFoundError(
        f"No KITTI images found under {DATA_ROOT / 'image_2'}. "
        "Follow docs/onboarding/seance-04-mini-kitti-reel.md first."
    )
if not 0 <= FRAME_INDEX < len(image_paths):
    raise IndexError(f"FRAME_INDEX={FRAME_INDEX} outside 0..{len(image_paths) - 1}")

image_path = image_paths[FRAME_INDEX]
sample_id = image_path.stem
lidar_path = DATA_ROOT / "velodyne" / f"{sample_id}.bin"
calibration_path = DATA_ROOT / "calib" / f"{sample_id}.txt"
for required_path in (lidar_path, calibration_path):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

print(f"sample={sample_id} ({FRAME_INDEX + 1}/{len(image_paths)})")
print(f"image={image_path}")
print(f"lidar={lidar_path}")

## Load and project

KITTI calibration composes the Velodyne-to-camera transform, rectification, and the camera-2 projection matrix. Points behind the camera or outside the image are removed by `project_lidar_to_image`.

In [ ]:
with Image.open(image_path) as image_file:
    rgb = np.asarray(image_file.convert("RGB"))

raw_points = load_velodyne_bin(lidar_path)
calibration = parse_kitti_calibration_file(calibration_path)
projected = project_lidar_to_image(
    raw_points,
    calibration,
    image_shape=rgb.shape[:2],
)

rounded_pixels = np.rint(projected.pixels).astype(np.int32)
unique_pixel_count = np.unique(rounded_pixels, axis=0).shape[0]
print(f"image shape: {rgb.shape[0]} x {rgb.shape[1]}")
print(f"raw LiDAR points: {raw_points.shape[0]:,}")
print(f"visible projected points: {projected.pixels.shape[0]:,}")
print(f"unique image pixels after rounding: {unique_pixel_count:,}")
print(f"projection retention: {100 * projected.pixels.shape[0] / raw_points.shape[0]:.1f}%")

In [ ]:
depth_m = projected.camera_points[:, 2]
fig, axes = plt.subplots(2, 1, figsize=(15, 8), constrained_layout=True)
axes[0].imshow(rgb)
axes[0].set_title(f"RGB - {sample_id}")
axes[0].axis("off")

axes[1].imshow(rgb)
points = axes[1].scatter(
    projected.pixels[:, 0],
    projected.pixels[:, 1],
    c=depth_m,
    s=POINT_SIZE,
    cmap="turbo",
    vmin=0.0,
    vmax=MAX_DEPTH_M,
    linewidths=0,
)
axes[1].set_title("Visible LiDAR points colored by camera depth")
axes[1].axis("off")
fig.colorbar(points, ax=axes[1], label="Depth (m)", fraction=0.025, pad=0.01)
plt.show()

## Projection diagnostics

The depth histogram exposes the useful range of the selected scan. The intensity plot is only a sensor-return diagnostic: intensity is not a material label and varies with range, angle, and sensor characteristics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
axes[0].hist(depth_m, bins=50, range=(0.0, MAX_DEPTH_M), color="tab:blue")
axes[0].set(title="Visible-point depth distribution", xlabel="Depth (m)", ylabel="Points")
axes[0].grid(alpha=0.25)

axes[1].scatter(depth_m, projected.intensity, s=2, alpha=0.35, color="tab:orange")
axes[1].set(title="LiDAR return intensity", xlabel="Camera depth (m)", ylabel="Intensity")
axes[1].set_xlim(0.0, MAX_DEPTH_M)
axes[1].grid(alpha=0.25)
plt.show()

## What to verify

Projected points should follow the road, vehicles, and visible structures. A global horizontal or vertical shift suggests an intrinsic/projection mismatch; a shape or orientation mismatch suggests an extrinsic-frame error. Correct projection here is a prerequisite for interpreting the six sparse channels in notebook `02_visualize_hybrid_input.ipynb`.